# SEA Avatar Scripts 


In [1]:
# SEA Avatar Script Generator — Single Notebook (LLM-driven, freeform)

# 1) Configure paths and run options
from pathlib import Path
from docx import Document

# Paths
BASE = Path('../00_API/SEA_Modules/en')           # Folder containing Module_1, Module_2, ...
MODULE_STRUCTURE = Path('../00_API/SEA_Modules/en/module_structure.json')  # Same level as Module_X
OUTPUT = Path('../03_Outputs/avatar_scripts')

# Which modules to process. Set to None for "all from module_structure.json". Example: ['1','2']
MODULES = None

# Model guidance for the LLM 
SYSTEM_STYLE = """You are an expert educational script writer.
Write a clear, engaging, speakable script for an on-screen avatar, for a video that will play at the beginning prior to the content. 

Use the provided source content. You may restructure freely. 

Keep it concise (2–3 minutes when spoken)."""


In [2]:
# 2) Azure OpenAI client setup 
import os
from dotenv import load_dotenv
load_dotenv()
AZURE_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

if not (AZURE_ENDPOINT and AZURE_API_KEY and AZURE_DEPLOYMENT):
    print("[WARN] Azure env missing. Set AZURE_OPENAI_ENDPOINT, AZURE_OPENAI_API_KEY, AZURE_OPENAI_DEPLOYMENT.")
else:
    print("[OK] Azure env detected.")


[OK] Azure env detected.


In [3]:
# 3) Minimal helpers — JSON loading and collection
import json, re
from typing import Dict, Any, List, Optional

def compact(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "")).strip()

def load_json(p: Path) -> Dict[str, Any]:
    try:
        return json.loads(p.read_text(encoding="utf-8"))
    except Exception:
        return {}

def find_module_folder(base: Path, mid: str) -> Optional[Path]:
    p = base / f"Module_{mid}"
    return p if p.exists() else None

def chapter_files(module_folder: Path, chapter_id_prefix: str) -> List[Path]:
    return sorted(module_folder.glob(f"{chapter_id_prefix}*.json"))

def collect_all_text(fp: Path) -> List[str]:
    j = load_json(fp)
    out = []
    for seg in j.get("segments", []):
        c = seg.get("content", {})
        if isinstance(c, dict):
            for k, v in c.items():
                if isinstance(v, str):
                    t = compact(v)
                    if t: out.append(t)
                elif isinstance(v, list):
                    for it in v:
                        if isinstance(it, dict):
                            for kk, vv in it.items():
                                if isinstance(vv, str):
                                    t = compact(vv)
                                    if t: out.append(t)
    return out

def module_title_from_structure(ms: Dict[str,Any], mid: str) -> Optional[str]:
    for m in ms.get("modules", []):
        if m.get("id") == mid:
            return compact(m.get("title", "")) or None
    return None

def chapter_title_from_structure(ms: Dict[str,Any], mid: str, cid: str) -> Optional[str]:
    for m in ms.get("modules", []):
        if m.get("id") == mid:
            for ch in m.get("chapters", []):
                if ch.get("id") == cid:
                    return compact(re.sub(r"<.*?>", "", ch.get("title", "")))
    return None

def chapter_ids_from_structure(ms: Dict[str,Any], mid: str) -> List[str]:
    out = []
    for m in ms.get("modules", []):
        if m.get("id") == mid:
            for ch in m.get("chapters", []):
                cid = ch.get("id")
                if cid not in (f"{mid}.0", f"{mid}.-1"):  # skip preface/outro
                    out.append(cid)
    return out

def chapter_number_from_structure(ms: Dict[str,Any], mid: str, cid: str) -> Optional[int]:
    for m in ms.get("modules", []):
        if m.get("id") == mid:
            chs = [ch for ch in m.get("chapters", [])
                   if ch.get("id") not in (f"{mid}.0", f"{mid}.-1")]
            for idx, ch in enumerate(chs, start=1):
                if ch.get("id") == cid:
                    return idx
    return None

def pretty_chapter_title(ms: Dict[str,Any], mid: str, cid: str) -> str:
    num = chapter_number_from_structure(ms, mid, cid)
    title = chapter_title_from_structure(ms, mid, cid) or ""
    if num:
        return f"Chapter {num}: {title}".strip(": ").strip()
    return f"Chapter: {title}".strip()

def save_docx(text: str, path: Path):
    doc = Document()
    for para in text.split("\n"):
        doc.add_paragraph(para)
    doc.save(path)



In [4]:
# 4) LLM call wrapper (Azure OpenAI)
from typing import List

def get_azure_client():
    try:
        from openai import AzureOpenAI
        return AzureOpenAI(
            api_key=AZURE_API_KEY,
            api_version=AZURE_API_VERSION,
            azure_endpoint=AZURE_ENDPOINT,
        )
    except Exception:
        from openai import OpenAI
        return OpenAI(
            api_key=AZURE_API_KEY,
            base_url=f"{AZURE_ENDPOINT}/openai",
            default_query={"api-version": AZURE_API_VERSION},
        )

def generate_script(client, system_prompt: str, desired_title: str, aggregated_text: List[str]) -> str:
    # Build the prompt — enforce the clean title
    content_block = "\n\n".join(aggregated_text[:2000])
    user_msg = f"""Write a speakable script that should start exactly with this title near the beginning:

{desired_title}

Rules:
- Begin with the title above, exactly like: Welcome to Module/Chapter 2: Title. Use numbers like 3 not three.
- This script should introduce this content that is about to come - don't just list the sections though, it should focus on key messages and data and statistics (only using data from the content provided).
- Write as a technical expert and communicator, avoid generalities or buzzwords. Avoid AI language like saying "it's not this, it is that"
- Please be as specific as possible. Feel free to include specific examples mentioned in the content where relevant. 
- Please limit background information and focus on the develompent opportunities found in the content. 
- Do not invent facts not in the source notes unless it's generic connective language.
- Utilize as many statistics and data points from the content as possible within the script, keeping the data and statistics well woven into the narrative. For any statistics as much as possible, provide the year of the statistic, so this script does not become outdated and incorrect in a few years. Don't say "currently" for statistics, provide the year.
- Avoid acronyms, as this will be narrated using AI.
- Provide narrative script only, not storyboard notes.
 Please use the punctuation marks as follows to mimic natural-sounding language: Hyphens (-): Separate syllables for clear pronunciation; Commas (,): Create shorter breaks; Periods (.): Introduce longer breaks with downward inflection.



Source notes:
{content_block}
"""

    if client is None:
        return f"[LLM DISABLED]\n\n{desired_title}\n\n" + content_block[:4000]

    try:
        resp = client.chat.completions.create(
            model=AZURE_DEPLOYMENT,  # your deployment name
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.7,
            max_tokens=900,
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[LLM ERROR: {e}]\n\n{desired_title}\n\n{content_block[:4000]}"


In [5]:
# 5) Main generation loop — modules + chapters (structure only, no fallback)
def run_pipeline():
    # load structure
    ms = load_json(MODULE_STRUCTURE)
    all_mids = [m.get("id") for m in ms.get("modules", [])]
    run_mids = MODULES or all_mids

    client = get_azure_client()
    OUTPUT.mkdir(parents=True, exist_ok=True)

    for mid in run_mids:
        module_folder = BASE / f"Module_{mid}"
        if not module_folder.exists():
            print(f"[WARN] Missing Module_{mid}")
            continue

        out_dir = OUTPUT / ("Module "+mid)
        out_dir.mkdir(parents=True, exist_ok=True)

        # --- module intro ---
        module_title = module_title_from_structure(ms, mid) or f"Module {mid}"
        module_texts = []
        for p in sorted(module_folder.glob('*.json')):
            module_texts.extend(collect_all_text(p))
        intro_title = f"Module {mid}: {module_title}"
        script = generate_script(client, SYSTEM_STYLE, intro_title, module_texts)
        print(script,"\n\n")
        save_docx(script, out_dir / f"{mid}.0.0-en.docx")
        
        # --- chapters ---
        cids = chapter_ids_from_structure(ms, mid)
        for cid in cids:
            agg = []
            for fp in chapter_files(module_folder, cid):
                agg.extend(collect_all_text(fp))

            # pretty title (uses ordinal + title from structure)
            desired_title = pretty_chapter_title(ms, mid, cid)

            ch_script = generate_script(client, SYSTEM_STYLE, desired_title, agg)
            print(ch_script,"\n\n")
            save_docx(ch_script, out_dir / f"{cid}-en.docx")

    print(f"[OK] Scripts written to {OUTPUT}")

# run
run_pipeline()


Welcome to Module 1: Intro to Sustainable Energy for Development.

Today, we begin a critical journey into understanding how energy shapes development and the pathways toward a sustainable future. Did you know that energy production and use account for approximately 73% of global greenhouse gas emissions, as of 2023? This makes our energy systems the single largest contributor to climate change.

Energy is fundamental to human well-being. Countries with higher levels of energy access typically have higher Human Development Index scores—highlighting a direct link between reliable energy and economic and social progress. Yet, nearly 800 million people worldwide still lack electricity, and over 2.4 billion rely on polluting fuels for cooking, primarily in low-income regions.

Access to clean, reliable energy also promotes gender equality. Women and girls often spend up to 18 hours weekly gathering firewood or cooking with inefficient fuels, limiting their opportunities in education and em

In [1]:
##translation for avatar scripts

import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from openai import AsyncAzureOpenAI
from docx import Document

# --- Load environment ---
load_dotenv()

# --- Config ---
INPUT_DIR = Path("../03_Outputs/final scripts/en")   # English source scripts
OUTPUT_DIR = Path("../03_Outputs/final scripts")     # Base output folder
TARGET_LANGUAGES = ["fr", "es", "pt", "ar", "zh", "ru"]

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")

# --- Translator Client ---
client = AsyncAzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
)

# --- Helpers ---
def read_docx(file_path: Path) -> str:
    doc = Document(file_path)
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

def write_docx(text: str, out_path: Path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    doc = Document()
    for line in text.split("\n"):
        doc.add_paragraph(line)
    doc.save(out_path)

async def translate_text(text: str, target_lang: str) -> str:
    """
    Translate text into one target language and return plain text.
    """
    prompt = (
        f"You are a professional translator. Preserve formatting and line breaks. "
        f"Translate the following text into {target_lang}. "
        f"Return ONLY the translated text, nothing else.\n\n"
        f"Text:\n{text}"
    )
    response = await client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        messages=[
            {"role": "system", "content": "You are a professional translator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0,
        max_tokens=4000
    )
    return response.choices[0].message.content.strip()

def read_docx(file_path: Path) -> str:
    try:
        doc = Document(file_path)
        return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])
    except Exception as e:
        print(f"⚠️ Skipping {file_path}, not a valid .docx ({e})")
        return ""


# --- Main Routine ---
async def translate_avatar_scripts():
    for file in INPUT_DIR.rglob("*.docx"):
        if "-en.docx" not in file.name:
            continue  # only process English scripts

        try:
            script_text = read_docx(file)
            if not script_text.strip():
                continue  # skip empty/invalid files
        except Exception as e:
            print(f"⚠️ Skipping {file}: {e}")
            continue
    
        script_text = read_docx(file)
        rel_path = file.relative_to(INPUT_DIR)  # e.g., Module_1/1.1-en.docx
        module_folder = rel_path.parent
        base_name = file.stem.replace("-en", "")

        print(f"📄 Translating {rel_path}...")

        for lang in TARGET_LANGUAGES:
            try:
                translated_text = await translate_text(script_text, lang)
            except Exception as e:
                print(f"⚠️ Failed {rel_path} → {lang}: {e}")
                continue

            out_dir = OUTPUT_DIR / lang / module_folder
            out_file = out_dir / f"{base_name}-{lang}.docx"
            write_docx(translated_text, out_file)
            print(f"  ✅ Saved {rel_path} → {lang}")

    print("🎉 All avatar translations complete.")


await translate_avatar_scripts()


📄 Translating Module 5/5.2.0-en.docx...
  ✅ Saved Module 5/5.2.0-en.docx → fr
  ✅ Saved Module 5/5.2.0-en.docx → es
  ✅ Saved Module 5/5.2.0-en.docx → pt
  ✅ Saved Module 5/5.2.0-en.docx → ar
  ✅ Saved Module 5/5.2.0-en.docx → zh
  ✅ Saved Module 5/5.2.0-en.docx → ru
📄 Translating Module 5/5.3.0-en.docx...
  ✅ Saved Module 5/5.3.0-en.docx → fr
  ✅ Saved Module 5/5.3.0-en.docx → es
  ✅ Saved Module 5/5.3.0-en.docx → pt
  ✅ Saved Module 5/5.3.0-en.docx → ar
  ✅ Saved Module 5/5.3.0-en.docx → zh
  ✅ Saved Module 5/5.3.0-en.docx → ru
📄 Translating Module 5/5.0.0-en.docx...
  ✅ Saved Module 5/5.0.0-en.docx → fr
  ✅ Saved Module 5/5.0.0-en.docx → es
  ✅ Saved Module 5/5.0.0-en.docx → pt
  ✅ Saved Module 5/5.0.0-en.docx → ar
  ✅ Saved Module 5/5.0.0-en.docx → zh
  ✅ Saved Module 5/5.0.0-en.docx → ru
📄 Translating Module 5/5.1.0-en.docx...
  ✅ Saved Module 5/5.1.0-en.docx → fr
  ✅ Saved Module 5/5.1.0-en.docx → es
  ✅ Saved Module 5/5.1.0-en.docx → pt
  ✅ Saved Module 5/5.1.0-en.docx → ar
  ✅ 